In [ ]:
# ------------------------------------------------------------------------
# RF-DETR
# Copyright (c) 2025 Roboflow. All Rights Reserved.
# Licensed under the Apache License, Version 2.0 [see LICENSE for details]
# ------------------------------------------------------------------------

# Query Embeddings for Dataset Quality Auditing

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/roboflow/rf-detr/blob/develop/docs/cookbooks/embedding-extraction.ipynb)

RF-DETR predicts objects with a set of *queries*. Each query carries a hidden-state vector from the last decoder
layer — the same vector the box and class heads read from. `predict(..., return_embeddings=True)` exposes that
vector for every returned detection, so you get one embedding per detected object instead of one embedding per
image.

**Why per-object embeddings matter.** Image-level embeddings (CLIP, DINOv2, a classifier backbone) describe a
whole scene. Quality problems in a detection dataset are *per annotation*: a box labelled `truck` that actually
contains a `bus`, an object nobody annotated, a duplicated instance. Those live at the object level, and RF-DETR
already computes an object-level representation as part of normal inference — this cookbook just reads it out.

**What this notebook demonstrates**, using a public COCO 2017 validation subset, no API key required:

| Section | Task | What it shows |
|---|---|---|
| 3 | Extract embeddings | `predict(..., return_embeddings=True)` -> `detections.data["embeddings"]`, shape `(K, H)` |
| 4 | Structure | Detections of the same category cluster together in a 2D projection |
| 5 | Probe | A k-NN probe recovers the ground-truth category from the embedding alone |
| 6 | **Mislabelled objects** | Label noise is injected into a known subset, then ranked back out (ROC-AUC, precision@k) |
| 7 | **Unannotated objects** | Confident detections with no ground-truth overlap are surfaced as missing labels |
| 8 | Similarity search | Nearest-neighbour retrieval of visually similar instances |
| 9 | Optimized models | How `inference(return_embeddings=True)` differs from the eager path |

Section 6 is the industrial-inspection use case from the feature request: we *know* which annotations were
corrupted, so the ranking quality is measured rather than eyeballed.

> **Runtime:** ~5 minutes on a Colab T4 with the default settings. It also runs on CPU (slower). Nothing here
> requires training.

## 1. Setup

`rfdetr` brings the model and `supervision` detections; `pyarrow` streams the dataset subset straight from the
Hugging Face parquet endpoint; `scikit-learn` provides the projection and the k-NN probe.

In [ ]:
!pip install -q "rfdetr>=1.10.0" pyarrow fsspec aiohttp scikit-learn matplotlib
# Until 1.10.0 is on PyPI, install the branch that adds the embedding interface instead:
!pip install -q "rfdetr @ git+https://github.com/roboflow/rf-detr.git@develop" pyarrow fsspec aiohttp scikit-learn matplotlib

In [ ]:
"""Extract per-detection RF-DETR query embeddings and use them to audit a detection dataset."""

import io
import warnings
from dataclasses import dataclass

import fsspec
import matplotlib.pyplot as plt
import numpy as np
import pyarrow.parquet as pq
import torch
from PIL import Image
from sklearn.decomposition import PCA
from sklearn.manifold import TSNE
from sklearn.metrics import roc_auc_score
from sklearn.neighbors import NearestNeighbors

from rfdetr import RFDETRSmall
from rfdetr.assets.coco_classes import COCO_CLASSES

warnings.filterwarnings("ignore", category=UserWarning)

SEED = 0
NUM_IMAGES = 300  # 100 images per parquet row group; raise for a denser embedding space
CONFIDENCE_THRESHOLD = 0.5
IOU_MATCH_THRESHOLD = 0.5  # detection <-> ground-truth association
IOU_UNANNOTATED_THRESHOLD = 0.3  # below this a detection covers no annotated object
NOISE_FRACTION = 0.10  # share of matched annotations whose label is corrupted in section 6
K_NEIGHBOURS = 10

rng = np.random.default_rng(SEED)
torch.manual_seed(SEED)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"device: {DEVICE}")

## 2. Load a public COCO validation subset

The subset is streamed from the Hugging Face parquet export of COCO 2017 (`rafaelpadilla/coco2017`). Only the
row groups we need are downloaded — 100 images each, roughly 16 MB — so this stays small and repeatable, and no
credentials are involved.

Its `label` field indexes the 91-entry COCO category table, i.e. the same raw COCO category ids RF-DETR's
COCO-pretrained checkpoints return as `class_id`. Ground truth and predictions are therefore directly comparable
without a remapping table.

In [ ]:
PARQUET_URL = "https://huggingface.co/api/datasets/rafaelpadilla/coco2017/parquet/default/val/0.parquet"


@dataclass
class Sample:
    """One dataset image with its ground-truth annotations.

    Attributes:
        image_id: COCO image id.
        image: Decoded RGB image.
        boxes: Ground-truth boxes in ``xyxy`` pixel coordinates, shape ``(N, 4)``.
        labels: Raw COCO category id per ground-truth box, shape ``(N,)``.
    """

    image_id: int
    image: Image.Image
    boxes: np.ndarray
    labels: np.ndarray


def load_coco_val_subset(url: str, num_images: int) -> list[Sample]:
    """Stream the first ``num_images`` COCO validation images from a parquet export.

    Only whole row groups are read, so the download stops as soon as enough images are collected.

    Args:
        url: HTTP(S) location of the parquet shard.
        num_images: Number of images to return.

    Returns:
        Decoded samples with ground-truth boxes in ``xyxy`` pixel coordinates.
    """
    samples: list[Sample] = []
    with fsspec.open(url) as file:
        parquet_file = pq.ParquetFile(file)
        for group_index in range(parquet_file.metadata.num_row_groups):
            if len(samples) >= num_images:
                break
            table = parquet_file.read_row_group(group_index)
            for row in table.to_pylist():
                if len(samples) >= num_images:
                    break
                objects = row["objects"]
                keep = [i for i, crowd in enumerate(objects["iscrowd"]) if not crowd]
                if not keep:
                    continue
                # COCO stores boxes as [x, y, width, height]; convert to xyxy.
                xywh = np.asarray([objects["bbox"][i] for i in keep], dtype=np.float32)
                boxes = np.column_stack([xywh[:, 0], xywh[:, 1], xywh[:, 0] + xywh[:, 2], xywh[:, 1] + xywh[:, 3]])
                samples.append(
                    Sample(
                        image_id=int(row["image_id"]),
                        image=Image.open(io.BytesIO(row["image"]["bytes"])).convert("RGB"),
                        boxes=boxes,
                        labels=np.asarray([objects["label"][i] for i in keep], dtype=np.int64),
                    )
                )
    return samples


samples = load_coco_val_subset(PARQUET_URL, NUM_IMAGES)
print(f"images: {len(samples)}")
print(f"ground-truth objects: {sum(len(s.labels) for s in samples)}")

## 3. Extract per-detection embeddings

This is the whole public interface: pass `return_embeddings=True` to `predict()`. The embeddings are gathered with
the same indices used for boxes, scores, and classes, so row `i` of `detections.data["embeddings"]` belongs to
detection `i` — no bookkeeping required on your side.

In [ ]:
model = RFDETRSmall()

detections_per_image = []
for start in range(0, len(samples), 8):
    batch = [sample.image for sample in samples[start : start + 8]]
    detections_per_image.extend(model.predict(batch, threshold=CONFIDENCE_THRESHOLD, return_embeddings=True))

first_with_objects = next(det for det in detections_per_image if len(det) > 0)
embeddings = first_with_objects.data["embeddings"]
print(f"detections in this image: {len(first_with_objects)}")
print(f"embeddings shape: {embeddings.shape}  (one row per detection, {embeddings.shape[1]}-d)")
print(f"embeddings dtype: {embeddings.dtype}")

## 4. Associate detections with annotations

To audit *annotations*, each detection is matched to the ground-truth box it covers (greedy, class-agnostic, by
IoU). Matched detections inherit the annotation's label and become the audit set. Unmatched confident detections
are kept aside for section 7.

In [ ]:
def iou_matrix(boxes_a: np.ndarray, boxes_b: np.ndarray) -> np.ndarray:
    """Compute the pairwise IoU between two sets of ``xyxy`` boxes.

    Args:
        boxes_a: Boxes with shape ``(N, 4)``.
        boxes_b: Boxes with shape ``(M, 4)``.

    Returns:
        IoU values with shape ``(N, M)``.
    """
    if len(boxes_a) == 0 or len(boxes_b) == 0:
        return np.zeros((len(boxes_a), len(boxes_b)), dtype=np.float32)
    top_left = np.maximum(boxes_a[:, None, :2], boxes_b[None, :, :2])
    bottom_right = np.minimum(boxes_a[:, None, 2:], boxes_b[None, :, 2:])
    overlap = np.clip(bottom_right - top_left, 0.0, None).prod(axis=2)
    area_a = (boxes_a[:, 2] - boxes_a[:, 0]) * (boxes_a[:, 3] - boxes_a[:, 1])
    area_b = (boxes_b[:, 2] - boxes_b[:, 0]) * (boxes_b[:, 3] - boxes_b[:, 1])
    return overlap / np.clip(area_a[:, None] + area_b[None, :] - overlap, 1e-9, None)


def match_to_ground_truth(detection_boxes: np.ndarray, gt_boxes: np.ndarray, iou_threshold: float) -> np.ndarray:
    """Greedily assign each detection to at most one ground-truth box.

    Pairs are consumed in descending IoU order, so a ground-truth box is claimed by its best detection only.

    Args:
        detection_boxes: Predicted boxes with shape ``(N, 4)`` in ``xyxy``.
        gt_boxes: Ground-truth boxes with shape ``(M, 4)`` in ``xyxy``.
        iou_threshold: Minimum IoU for a valid match.

    Returns:
        Index of the matched ground-truth box per detection, or ``-1`` where unmatched, shape ``(N,)``.
    """
    ious = iou_matrix(detection_boxes, gt_boxes)
    assignment = np.full(len(detection_boxes), -1, dtype=np.int64)
    if ious.size == 0:
        return assignment
    taken_gt: set[int] = set()
    order = np.dstack(np.unravel_index(np.argsort(ious, axis=None)[::-1], ious.shape))[0]
    for detection_index, gt_index in order:
        if ious[detection_index, gt_index] < iou_threshold:
            break
        if assignment[detection_index] != -1 or gt_index in taken_gt:
            continue
        assignment[detection_index] = gt_index
        taken_gt.add(int(gt_index))
    return assignment


matched_embeddings: list[np.ndarray] = []
matched_labels: list[int] = []  # ground-truth category id
matched_sources: list[tuple[int, int]] = []  # (sample index, detection index)
unannotated_sources: list[tuple[int, int]] = []

for sample_index, (sample, detections) in enumerate(zip(samples, detections_per_image)):
    if len(detections) == 0:
        continue
    assignment = match_to_ground_truth(detections.xyxy, sample.boxes, IOU_MATCH_THRESHOLD)
    best_iou = iou_matrix(detections.xyxy, sample.boxes).max(axis=1, initial=0.0)
    for detection_index, gt_index in enumerate(assignment):
        if gt_index >= 0:
            matched_embeddings.append(detections.data["embeddings"][detection_index])
            matched_labels.append(int(sample.labels[gt_index]))
            matched_sources.append((sample_index, detection_index))
        elif best_iou[detection_index] < IOU_UNANNOTATED_THRESHOLD:
            unannotated_sources.append((sample_index, detection_index))

embeddings = np.stack(matched_embeddings)
labels = np.asarray(matched_labels)
# Cosine similarity is the natural metric here, so work with unit-norm rows throughout.
unit_embeddings = embeddings / np.linalg.norm(embeddings, axis=1, keepdims=True)

print(f"matched detections (audit set): {len(embeddings)}")
print(f"confident detections without any annotation: {len(unannotated_sources)}")
print(f"distinct categories: {len(np.unique(labels))}")

## 5. Do the embeddings carry object semantics?

Two checks before relying on them. First a 2D projection (PCA to 50 dimensions, then t-SNE) coloured by the
ground-truth category: same-category detections should land together. Second a leave-one-out k-NN probe: if a
detection's neighbours predict its category well above the majority-class baseline, the geometry is meaningful,
which is exactly what the audit in section 6 depends on.

In [ ]:
top_categories = [int(c) for c, _ in sorted(zip(*np.unique(labels, return_counts=True)), key=lambda p: -p[1])[:8]]
plot_mask = np.isin(labels, top_categories)

projection = PCA(n_components=min(50, unit_embeddings.shape[1]), random_state=SEED).fit_transform(unit_embeddings)
projected_2d = TSNE(n_components=2, init="pca", perplexity=30, random_state=SEED).fit_transform(projection)

plt.figure(figsize=(8, 6))
for category in top_categories:
    category_mask = labels == category
    plt.scatter(
        projected_2d[category_mask, 0], projected_2d[category_mask, 1], s=14, alpha=0.75, label=COCO_CLASSES[category]
    )
plt.legend(loc="best", fontsize=8)
plt.title(f"RF-DETR query embeddings ({int(plot_mask.sum())} detections, t-SNE)")
plt.xticks([])
plt.yticks([])
plt.tight_layout()
plt.show()

In [ ]:
neighbours = NearestNeighbors(n_neighbors=K_NEIGHBOURS + 1, metric="cosine").fit(unit_embeddings)
_, neighbour_indices = neighbours.kneighbors(unit_embeddings)
neighbour_indices = neighbour_indices[:, 1:]  # drop self-match

neighbour_labels = labels[neighbour_indices]
knn_prediction = np.asarray([np.bincount(row).argmax() for row in neighbour_labels])
knn_accuracy = float((knn_prediction == labels).mean())
majority_baseline = float(np.bincount(labels).max() / len(labels))

print(f"{K_NEIGHBOURS}-NN leave-one-out accuracy: {knn_accuracy:.3f}")
print(f"majority-class baseline:      {majority_baseline:.3f}")

## 6. Find mislabelled annotations

The industrial-inspection scenario: an operator mislabels some objects and you want the review queue ordered so
the worst annotations surface first.

To *measure* this rather than guess, we corrupt a known 10% of the matched annotations by reassigning a random
different category. Each annotation is then scored by how much its own label disagrees with its neighbourhood in
embedding space — the neighbours are visually similar objects, so a label they do not share is suspicious:

```
suspicion = 1 - (similarity-weighted share of the k nearest neighbours carrying the same label)
```

Nothing about which annotations were corrupted enters the score; the corruption mask is only used afterwards to
grade the ranking with ROC-AUC and precision@k.

In [ ]:
noisy_labels = labels.copy()
category_pool = np.unique(labels)
num_corrupted = int(NOISE_FRACTION * len(labels))
corrupted_indices = rng.choice(len(labels), size=num_corrupted, replace=False)
for index in corrupted_indices:
    alternatives = category_pool[category_pool != labels[index]]
    noisy_labels[index] = rng.choice(alternatives)

is_corrupted = np.zeros(len(labels), dtype=bool)
is_corrupted[corrupted_indices] = True
print(f"corrupted annotations: {is_corrupted.sum()} / {len(labels)}")

In [ ]:
neighbour_similarity = 1.0 - neighbours.kneighbors(unit_embeddings)[0][:, 1:]
agreement = (noisy_labels[neighbour_indices] == noisy_labels[:, None]).astype(np.float32)
suspicion = 1.0 - (agreement * neighbour_similarity).sum(axis=1) / np.clip(neighbour_similarity.sum(axis=1), 1e-9, None)

ranking = np.argsort(suspicion)[::-1]
roc_auc = roc_auc_score(is_corrupted, suspicion)
review_budget = num_corrupted
precision_at_k = float(is_corrupted[ranking[:review_budget]].mean())
random_precision = float(is_corrupted.mean())

print(f"ROC-AUC of the suspicion score:      {roc_auc:.3f}")
print(f"precision@{review_budget} (embedding ranking): {precision_at_k:.3f}")
print(f"precision@{review_budget} (random review):     {random_precision:.3f}")
print(f"lift over random review: {precision_at_k / max(random_precision, 1e-9):.1f}x")

Reading the numbers: ROC-AUC is the probability that a corrupted annotation outranks a clean one, and
precision@k is the share of genuinely bad annotations in a review queue the size of the corruption. Compare it
against the random-review baseline — that gap is the labour the embeddings save.

The plot below shows the score distributions, and the grid shows the top-ranked suspects with their (possibly
corrupted) label and the label their neighbourhood votes for.

In [ ]:
plt.figure(figsize=(8, 4))
plt.hist(suspicion[~is_corrupted], bins=40, alpha=0.7, density=True, label="clean annotations")
plt.hist(suspicion[is_corrupted], bins=40, alpha=0.7, density=True, label="corrupted annotations")
plt.xlabel("suspicion score")
plt.ylabel("density")
plt.title(f"Label-error ranking (ROC-AUC {roc_auc:.3f})")
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
def crop_detection(index: int, padding: float = 0.06) -> Image.Image:
    """Crop the image region of an audited detection.

    Args:
        index: Row index into the matched-detection arrays.
        padding: Fraction of the box size added on every side for context.

    Returns:
        The cropped RGB region.
    """
    sample_index, detection_index = matched_sources[index]
    sample = samples[sample_index]
    x1, y1, x2, y2 = detections_per_image[sample_index].xyxy[detection_index]
    pad_x, pad_y = padding * (x2 - x1), padding * (y2 - y1)
    box = (
        max(0.0, x1 - pad_x),
        max(0.0, y1 - pad_y),
        min(float(sample.image.width), x2 + pad_x),
        min(float(sample.image.height), y2 + pad_y),
    )
    return sample.image.crop(box)


neighbour_vote = np.asarray([np.bincount(noisy_labels[row]).argmax() for row in neighbour_indices])

figure, axes = plt.subplots(2, 5, figsize=(15, 7))
for axis, index in zip(axes.ravel(), ranking[:10]):
    axis.imshow(crop_detection(index))
    verdict = "injected error" if is_corrupted[index] else "flagged"
    axis.set_title(
        f"label: {COCO_CLASSES[noisy_labels[index]]}\nneighbours: {COCO_CLASSES[neighbour_vote[index]]}\n{verdict}",
        fontsize=9,
    )
    axis.axis("off")
figure.suptitle("Top-ranked annotation suspects")
figure.tight_layout()
plt.show()

Suspects that are *not* injected errors are worth a look rather than a dismissal: they are typically ambiguous
crops, near-duplicate categories (`car` vs `truck`), or annotations that were already questionable in the
original dataset.

## 7. Find unannotated objects

The same pass also finds the opposite failure mode. A confident detection that overlaps no annotation is either a
false positive or an object nobody labelled — the second is a dataset bug that silently penalises training and
evaluation. Grouping the candidates by nearest neighbours among *audited* detections tells you what the model
thinks they are.

In [ ]:
unannotated_sorted = sorted(
    unannotated_sources,
    key=lambda source: float(detections_per_image[source[0]].confidence[source[1]]),
    reverse=True,
)

figure, axes = plt.subplots(2, 5, figsize=(15, 7))
for axis, (sample_index, detection_index) in zip(axes.ravel(), unannotated_sorted[:10]):
    sample = samples[sample_index]
    detections = detections_per_image[sample_index]
    x1, y1, x2, y2 = detections.xyxy[detection_index]
    axis.imshow(sample.image.crop((float(x1), float(y1), float(x2), float(y2))))
    axis.set_title(
        f"{COCO_CLASSES[int(detections.class_id[detection_index])]}"
        f" ({detections.confidence[detection_index]:.2f})\nno annotation",
        fontsize=9,
    )
    axis.axis("off")
figure.suptitle("Confident detections with no matching annotation")
figure.tight_layout()
plt.show()

## 8. Similarity search over objects

Because the embeddings live in a shared space, a single detection can be used as a query to retrieve visually
similar instances across the dataset — the building block for near-duplicate removal, re-identification, and
"show me more objects like this one" review tooling.

In [ ]:
query_index = int(ranking[0])
_, retrieved = neighbours.kneighbors(unit_embeddings[query_index : query_index + 1], n_neighbors=6)

figure, axes = plt.subplots(1, 6, figsize=(16, 3.2))
for rank, (axis, index) in enumerate(zip(axes, retrieved[0])):
    axis.imshow(crop_detection(int(index)))
    axis.set_title("query" if rank == 0 else f"#{rank} · {COCO_CLASSES[labels[index]]}", fontsize=9)
    axis.axis("off")
figure.suptitle("Nearest neighbours of a query object")
figure.tight_layout()
plt.show()

## 9. Embeddings from an optimized model

`model.inference()` traces/compiles the forward pass, and a traced graph has fixed control flow: whether
embeddings are produced cannot be decided per call any more. Pass the flag to `inference()` and keep `predict()`
consistent with it — a mismatch raises `RuntimeError` instead of silently returning nothing.

```python
model = RFDETRSmall()
model.inference(return_embeddings=True)  # decided once, at optimization time

detections = model.predict(image, threshold=0.5, return_embeddings=True)
detections.data["embeddings"]  # (K, H)
```

The eager path used throughout this notebook has no such constraint — `return_embeddings` is a plain per-call
argument.

Segmentation models behave identically. Keypoint models attach embeddings to `key_points.data["embeddings"]`,
since they return `sv.KeyPoints` rather than `sv.Detections`.

## 10. Takeaways

- `predict(..., return_embeddings=True)` returns one embedding per detection in `detections.data["embeddings"]`,
  aligned row-for-row with boxes, scores, and classes.
- They are the decoder hidden states the detection heads already consume, so they cost nothing extra to obtain
  and require no second model.
- Neighbourhood disagreement in that space ranks mislabelled annotations far above random review, and detections
  without ground-truth overlap surface missing annotations — both measured above on a public dataset.
- The same vectors support similarity search, near-duplicate detection, clustering, and active-learning
  selection.

**Next steps:** run this over your own dataset with a fine-tuned checkpoint (`RFDETRSmall(pretrain_weights=...)`),
and route the top-ranked suspects into your labelling tool as a review queue.